In [1]:
from common import getBatch, downloadDataset
from tqdm import tqdm
import numpy as np
from typing import Tuple
import os

/mnt/c/Users/Usuario UTP/Documents/tareas/redNeuronal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from typing import List, Callable, Any
def sigmoidea(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def devSigmoidea(x: np.ndarray) -> np.ndarray:
    s = sigmoidea(x)
    return s * (1.0 - s)

def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)

def devRelu(x: np.ndarray) -> np.ndarray:
    return np.where(x > 0, 1.0, 0.0)

def softmax(x: np.ndarray) -> np.ndarray:
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def devSoftmax(x: np.ndarray) -> np.ndarray:
    s = softmax(x)
    s_vec = s.reshape(-1)
    jacobian_matrix = np.diag(s_vec) - np.outer(s_vec, s_vec)
    return jacobian_matrix

def mse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    return np.mean((predicted - actually) ** 2)

def devMse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    n = predicted.size
    return np.where(n > 0, (2.0 / n) * (predicted - actually), np.zeros_like(predicted))

def lostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -np.sum(actually * np.log(p))

def devLostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -(actually / p)

In [3]:
class Layer:
    def __init__(
            self,
            neurons: int,
            activation: Callable[[np.ndarray], np.ndarray],
            derivada: Callable[[np.ndarray], np.ndarray],
            name: str):
        self.neurons: int = neurons
        self.activacion = activation
        self.derivada = derivada
        self.name: str = name
        
    def export(self)->str:
        return f"{self.name}-{self.neurons}-{self.activacion.__name__}-{self.derivada.__name__}"
    
    @staticmethod
    def load(layer: str)->'Layer':
        name, neurons_str, activationName, derivadaName = layer.split("-")
        neurons = int(neurons_str)
        activation = None
        derivada = None
        match(activationName):
            case "sigmoidea":
                activation = sigmoidea
            case "relu":
                activation = relu
            case "softmax":
                activation = softmax
        
        match(derivadaName):
            case "devSigmoidea":
                derivada = devSigmoidea
            case "devRelu":
                derivada = devRelu
            case "devSoftmax":
                derivada = devSoftmax
        return Layer(neurons, activation, derivada, name)
        

class Model:
    def __init__(
            self,
            sequential: Any,
            w: List[np.ndarray],
            b: List[np.ndarray]):
        self.sequential = sequential
        self.set_parameters(w, b)
        
    def set_parameters(self, w: List[np.ndarray], b: List[np.ndarray]):
        self.w = [np.array(weights, dtype=np.float32) for weights in w]
        self.b = [np.array(bias, dtype=np.float32) for bias in b]
        
    def getParams(self):
        return (self.w, self.b)

    def fordward(self, x: np.ndarray) -> np.ndarray:
      neu = [None] * len(self.sequential)
      z = [None] * len(self.sequential)
      neu[0] = x
      for i in range(1, len(self.sequential)):
        neu[i] = self.sequential[i].activacion(np.dot(neu[i - 1], self.w[i - 1]) + self.b[i])
      return neu[-1]

class Sequential:
    def __init__(self, *layers: Layer):
        self.layers = layers

    def __len__(self):
        return len(self.layers)

    def __getitem__(self, i):
        return self.layers[i]
    
    def export(self):
        export = ""
        nLayer = len(self.layers)
        for index, i in enumerate(self.layers):
            if index < nLayer - 1:
                export += f"{i.export()}\n"
            else:
                export += f"{i.export()}"
        return export
    
    @staticmethod
    def load(layers: str)->'Sequential':
        internal = []
        for i in layers.split("\n"):
            internal.append(Layer.load(i))
        return Sequential(*internal)


In [14]:
def __evaluate(neu, x_sample: np.ndarray, z, sequential, w, b):
    neu[0] = x_sample
    for i in range(1, len(sequential)):
        z[i] = np.dot(neu[i - 1], w[i - 1]) + b[i]
        neu[i] = sequential[i].activacion(z[i])

def evaluate(w, b, x_test, y_test, sequential, error):
    correct_predictions = 0
    errors = []
    for x_b, y_b in zip(x_test, y_test):
        neu_v = [None] * len(sequential)
        z_v = [None] * len(sequential)
        __evaluate(neu_v, x_b, z_v, sequential, w, b)
        errors.append(error(neu_v[-1], y_b))
        if np.argmax(neu_v[-1]) == np.argmax(y_b):
            correct_predictions += 1
    return (correct_predictions/max(x_test.shape[0], 1), errors)

In [15]:
from multiprocessing import Process, Pipe
import psutil
import os
import traceback
import time
import math

def __forward(neu, x_sample: np.ndarray, z, sequential, w, b):
    neu[0] = x_sample
    for i in range(1, len(sequential)):
        z[i] = np.dot(neu[i - 1], w[i - 1]) + b[i]
        neu[i] = sequential[i].activacion(z[i])

def __backward(dEdz, z, sequential, w) -> np.ndarray:
    for i in range(len(sequential) - 2, 0, -1):
        dEdz[i] = (dEdz[i+1] @ w[i].T) * sequential[i].derivada(z[i])
        
def batch(x_b, y_b, w, b, w_grad_batch, b_grad_batch, sequential, devError):
    neu = [None] * len(sequential)
    z = [None] * len(sequential)
    __forward(neu, x_b, z, sequential, w, b)
    dEdz = [None] * len(sequential)
    de = devError(neu[-1], y_b)
    if de.shape == (1,):
      dEdz[-1] = de * sequential[-1].derivada(z[-1])
    else:
      dEdz[-1] = de @ sequential[-1].derivada(z[-1])
    __backward(dEdz, z, sequential, w)
    for i in range(len(sequential) - 1):
        w_grad_sample = np.outer(neu[i], dEdz[i+1])
        w_grad_batch[i] += w_grad_sample
        b_grad_batch[i+1] += dEdz[i+1]

In [16]:
from socket import socket
from typing import Dict, Any
import pickle
import json
import shutil
import os

def recvall(sock: socket, n: int) -> bytearray:
    data = bytearray()
    while len(data) < n:
        packet = sock.recv(n - len(data))
        if not packet:
            raise ConnectionError("Conexión cerrada antes de recibir todos los datos esperados")
        data.extend(packet)
    return data

class State:
    
    def __init__(self, params: Dict[str, Any]):
        self.data = params
    
    def do(self, sock: socket):
        self.data["status"] = "ok"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return HandShake(self.data)

class HandShake(State):
    
    def do(self, sock: socket):
        super().do(sock)
        try:
            dataLen = recvall(sock, 8)
            data = recvall(sock, int.from_bytes(dataLen, 'big')).decode("utf-8")
            self.data.update(json.loads(data))
            self.data["shape"] = tuple(self.data["shape"])
            seed = int(self.data["seed"])
            os.environ["token"] = self.data["token"]
            self.data["sequential"] = Sequential.load(self.data["sequential"])
            match self.data["error"]:
                case "mse":
                    self.data["error"] = mse
                case "lostEntropy":
                    self.data["error"] = lostEntropy
            match self.data["devError"]:
                case "devMse":
                    self.data["devError"] = devMse
                case "devLostEntropy":
                    self.data["devError"] = devLostEntropy
        except Exception as e:
            print("HandShake error: ", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return Recolection(self.data)
    
class Recolection(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            dsName = self.data["dsName"]
            split = self.data["split"]
            (ds, size) = downloadDataset(dsName, split)
            self.data["ds"] = ds
            self.data["dsSize"] = size
            self.data["shard"] = int(size/(self.data["batchSize"]*self.data["workers"]))
            self.data["dsSize"] = self.data["datasetPorcent"]*self.data["dsSize"] / self.data["workers"]
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
        except Exception as e:
            print("Recolection error: ", e)
            message = f"n-{shardPosition}"
            try:
                sock.sendall(message.encode("utf-8"))
            except:
                pass
            sock.close()
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        return TrainBatch(self.data)

class TrainBatch(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            workers = self.data["workers"]
            length_prefix = recvall(sock, 8)
            message_length = int.from_bytes(length_prefix, 'big')
            data = recvall(sock, message_length)
            self.data["w"], self.data["b"] = pickle.loads(data)
            ds = self.data["ds"]
            batchSize = self.data["batchSize"]
            labels = self.data["labels"]
            size = self.data["dsSize"]
            w_grad_batch = [np.zeros_like(wi) for wi in self.data["w"]]
            b_grad_batch = [np.zeros_like(bi) for bi in self.data["b"]]
            sequential = self.data["sequential"]
            devError = self.data["devError"]
            split = self.data["split"]
            shard = self.data["shard"]
            if "batch_gen" not in self.data:
                self.data["batch_gen"] = getBatch(
                            ds, 
                            batchSize, 
                            labels, 
                            size, 
                            workers, 
                            (shard, shardPosition), 
                            shape=self.data["shape"], 
                            tqdmDisable=False,
                            classNumber=self.data["labelsNumber"]
                        )
            try:
                self.data["x"], self.data["y"] = next(self.data["batch_gen"])
                for x_b, y_b in zip(self.data["x"], self.data["y"]):
                    batch(x_b, y_b, self.data["w"], self.data["b"], w_grad_batch, b_grad_batch, sequential, devError)
            except StopIteration:
                del self.data["batch_gen"]
                
            self.data["w_grad_batch"] = w_grad_batch
            self.data["b_grad_batch"] = b_grad_batch
        except Exception as e:
            print("TrainBatch error: ", e)
            self.data["status"] = "error"
                
    def next(self):
        if self.data.get("status") == "error":
            return None
        if self.data["verbose"]:
            return Validate(self.data)
        return End(self.data)

class Validate(State):
    
    def do(self, sock: socket):
        super().do(sock) # No olvides inicializar el estado como "ok"
        try:
            (accuracy, batch_losses) = evaluate(
                self.data["w"], 
                self.data["b"], 
                self.data["x"], 
                self.data["y"], 
                self.data["sequential"], 
                self.data["error"]
            )
            test_accuracy = 0
            # if self.data.get("test"):
            #     (test_accuracy, _) = evaluate(
            #         self.data["w"], 
            #         self.data["b"], 
            #         self.data["x_test"], 
            #         self.data["y_test"], 
            #         self.data["sequential"], 
            #         self.data["error"]
            #     )
            datos_metricas = pickle.dumps((accuracy, test_accuracy, batch_losses))
            sock.sendall(len(datos_metricas).to_bytes(8, 'big'))
            sock.sendall(datos_metricas)
            
        except Exception as e:
            print("Validate error: ", e)
            self.data["status"] = "error"
    
    def next(self):
        if self.data.get("status") == "error":
            return None
        return End(self.data)

class End(State):
    
    def do(self, sock: socket):
        super().do(sock)
        shardPosition = self.data.get("shardPosition")
        try:
            message = f"y-{shardPosition}"
            sock.sendall(message.encode("utf-8"))
            w_grad_batch = self.data.get("w_grad_batch", [])
            b_grad_batch = self.data.get("b_grad_batch", [])
            data_to_send = pickle.dumps((w_grad_batch, b_grad_batch))
            sock.sendall(len(data_to_send).to_bytes(8, 'big'))
            sock.sendall(data_to_send)
            continueWork = recvall(sock, 1).decode("utf-8")
            self.data["continue"] = continueWork
        except Exception as e:
            print("End error: ", e)
            self.data["status"] = "error"
            
    def next(self):
        if self.data.get("status") == "error":
            return None
        match self.data.get("continue"):
            case "y":
                return TrainBatch(self.data)
            case "n":
                for folder in os.listdir('.'):
                    if folder.startswith('data-') and os.path.isdir(folder):
                        shutil.rmtree(folder)
                return None
            case _:
                return None


In [17]:
def __fit(host, port):
    import socket
    estado_actual = State({})
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.connect((HOST, PORT))
            print(f"Conectado a {HOST}:{PORT}")
            while estado_actual is not None:
                estado_actual.do(s)
                estado_actual = estado_actual.next()
        except Exception as e:
            print(e)
            s.close()

In [19]:
HOST = "127.0.0.1"
PORT = 65432
__fit(HOST, PORT)

Conectado a 127.0.0.1:65432


batch:   0%|          | 0/28 [00:01<?, ?it/s]
